## 1. импорты

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

# Ноутбук лежит в lab_2/notebooks, поэтому корень работы - папка выше.
ROOT = Path("..").resolve()
VPO_ROOT = ROOT / "_extracted"

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

pio.templates.default = "plotly_white"

COLORS = {
    "red": "#b42318",
    "blue": "#2f5d8c",
    "teal": "#0e7c7b",
    "green": "#4f7d45",
    "gold": "#c48a2c",
    "purple": "#7b61a8",
    "gray": "#667085",
}

ROOT

PosixPath('/home/knyze/big_data_analyze/lab_2')

## 2. Подготовка данных из исходных Excel-файлов

In [ ]:
def clean_text(value) -> str:
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\n", " ")).strip().lower()


def num(value) -> float:
    if pd.isna(value) or value == "":
        return 0.0
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).replace("\xa0", "").replace(" ", "").replace(",", ".")
    try:
        return float(text)
    except ValueError:
        return 0.0


def academic_year_start(year_code: str) -> int:
    return 2000 + int(year_code[:2])


def academic_year_label(year_code: str) -> str:
    start, end = year_code.split("-")
    return f"20{start}/20{end}"


year_codes = sorted(
    p.name
    for p in ROOT.iterdir()
    if p.is_dir() and re.fullmatch(r"\d{2}-\d{2}", p.name)
)
year_codes

['18-19', '19-20', '20-21', '21-22', '22-23', '23-24', '24-25', '25-26']

### 2.1. Школы: форма ОО-1

In [ ]:
def column_number_row(df: pd.DataFrame) -> int:
    for i in range(25):
        row = [clean_text(v) for v in df.iloc[i]]
        if "1" in row and "2" in row and "3" in row:
            return i
    raise ValueError("Не найдена строка с номерами граф")


def code_column(df: pd.DataFrame) -> int:
    row = [clean_text(v) for v in df.iloc[column_number_row(df)]]
    return row.index("2")


def row_by_code(df: pd.DataFrame, code: int) -> int:
    start = column_number_row(df) + 1
    col = code_column(df)
    for i in range(start, len(df)):
        if str(df.iat[i, col]).strip() == str(code):
            return i
    raise ValueError(f"Не найдена строка с кодом {code}")


def school_grade_columns(df: pd.DataFrame) -> dict[str, int]:
    columns = {}
    for j in range(df.shape[1]):
        header = " ".join(clean_text(df.iat[i, j]) for i in range(min(len(df), 25)))
        if "9-й класс" in header:
            columns["grade_9"] = j
        if "10-й класс" in header and "10–11" not in header and "10-11" not in header:
            columns["grade_10"] = j
        if "11-й класс (выпускной)" in header:
            columns["grade_11"] = j
    return columns


def total_classes_column(df: pd.DataFrame) -> int:
    for j in range(df.shape[1]):
        header = " ".join(clean_text(df.iat[i, j]) for i in range(min(len(df), 22)))
        if "итого по всем классам" in header:
            return j
    raise ValueError("Не найдена итоговая графа")


def read_school_file(path: Path) -> dict[str, float]:
    result = {
        "grade_9_students": 0.0,
        "grade_10_students": 0.0,
        "grade_11_students": 0.0,
        "grade_9_certificates": 0.0,
        "grade_9_continue_same_org": 0.0,
        "grade_11_certificates": 0.0,
    }

    xl = pd.ExcelFile(path)
    for sheet in xl.sheet_names:
        if "2.1.1" in sheet:
            df = pd.read_excel(path, sheet_name=sheet, header=None)
            cols = school_grade_columns(df)
            row = row_by_code(df, 10)
            result["grade_9_students"] += num(df.iat[row, cols["grade_9"]])
            result["grade_10_students"] += num(df.iat[row, cols["grade_10"]])
            result["grade_11_students"] += num(df.iat[row, cols["grade_11"]])

        if "2.6" in sheet:
            df = pd.read_excel(path, sheet_name=sheet, header=None)
            total_col = total_classes_column(df)
            result["grade_9_certificates"] += num(df.iat[row_by_code(df, 2), total_col])
            result["grade_9_continue_same_org"] += num(
                df.iat[row_by_code(df, 3), total_col]
            )
            result["grade_11_certificates"] += num(
                df.iat[row_by_code(df, 10), total_col]
            )

    return result

In [ ]:
school_rows = []

for year_code in year_codes:
    folders = [ROOT / year_code, VPO_ROOT / year_code]
    files = []
    for folder in folders:
        files.extend(folder.glob("*Россия*.xlsx"))

    for path in sorted(set(files)):
        name = path.name
        sector = "ГОУ+НОУ" if "ГОУ+НОУ" in name else "ГОУ" if "ГОУ" in name else "НОУ"
        settlement = (
            "город+село"
            if "город+село" in name
            else "город"
            if "город" in name
            else "село"
        )
        school_rows.append(
            {
                "year_code": year_code,
                "academic_year": academic_year_label(year_code),
                "year_start": academic_year_start(year_code),
                "sector": sector,
                "settlement": settlement,
                "source_file": str(path.relative_to(ROOT)),
                **read_school_file(path),
            }
        )

school = pd.DataFrame(school_rows).sort_values(["sector", "settlement", "year_start"])
school["same_school_continuation_pct"] = (
    school["grade_9_continue_same_org"] / school["grade_9_certificates"] * 100
)
school["prev_grade_9_students"] = school.groupby(["sector", "settlement"])[
    "grade_9_students"
].shift(1)
school["cohort_grade10_vs_prev_grade9_pct"] = (
    school["grade_10_students"] / school["prev_grade_9_students"] * 100
)

school_total = school.query("sector == 'ГОУ+НОУ' and settlement == 'город+село'").copy()
school_total.tail()

,year_code,academic_year,year_start,sector,settlement,source_file,grade_9_students,grade_10_students,grade_11_students,grade_9_certificates,grade_9_continue_same_org,grade_11_certificates,same_school_continuation_pct,prev_grade_9_students,cohort_grade10_vs_prev_grade9_pct
19,21-22,2021/2022,2021,ГОУ+НОУ,город+село,21-22/ОО-1_Россия_(ГОУ+НОУ)_(город+село).xlsx,"1,464,572.00","700,761.00","643,823.00","1,393,805.00","568,449.00","675,623.00",40.78,"1,420,507.00",49.33
31,22-23,2022/2023,2022,ГОУ+НОУ,город+село,22-23/Россия_(ГОУ+НОУ)_(город+село).xlsx,"1,589,478.00","686,538.00","619,517.00","1,419,901.00","560,369.00","631,945.00",39.47,"1,464,572.00",46.88
40,23-24,2023/2024,2023,ГОУ+НОУ,город+село,23-24/Россия_(ГОУ+НОУ)_(город+село).xlsx,"1,689,558.00","719,073.00","605,111.00","1,527,664.00","583,859.00","606,147.00",38.22,"1,589,478.00",45.24
49,24-25,2024/2025,2024,ГОУ+НОУ,город+село,24-25/Россия_(ГОУ+НОУ)_(город+село).xlsx,"1,741,257.00","738,721.00","632,090.00","1,622,200.00","605,428.00","593,616.00",37.32,"1,689,558.00",43.72
58,25-26,2025/2026,2025,ГОУ+НОУ,город+село,25-26/Россия_(ГОУ+НОУ)_(город+село).xlsx,"1,772,997.00","742,835.00","651,580.00","1,671,774.00","618,690.00","619,903.00",37.01,"1,741,257.00",42.66


### 2.2. Колледжи: форма СПО-1

In [ ]:
def spo_total_file(year_code: str) -> Path | None:
    files = []
    for folder in [ROOT / year_code, VPO_ROOT / year_code]:
        files.extend(folder.rglob("*СПО1*ВСЕГО*.xls*"))
    bad_words = ["очная", "заочная", "очно", "персонал", "аттеста"]
    files = [p for p in files if not any(word in p.name.lower() for word in bad_words)]
    return sorted(files, key=lambda p: (len(str(p)), str(p)))[0] if files else None


def read_spo_file(path: Path) -> dict[str, float]:
    sheet = next(
        s
        for s in pd.ExcelFile(path).sheet_names
        if clean_text(s).replace("p", "р") == "р2_1_1"
    )
    df = pd.read_excel(path, sheet_name=sheet, header=None)
    result = {
        "spo_applications_after_9": 0.0,
        "spo_admitted_after_9": 0.0,
        "spo_budget_after_9": 0.0,
        "spo_paid_after_9": 0.0,
        "spo_admitted_specialists_after_9": 0.0,
        "spo_admitted_workers_after_9": 0.0,
    }

    for _, row in df.iterrows():
        label = clean_text(row.iloc[0])
        if "на базе основного общего образования" not in label or "всего" not in label:
            continue

        is_specialist = "специалистов среднего звена" in label
        is_worker = "квалифицированных рабочих" in label or "служащих" in label
        if not (is_specialist or is_worker):
            continue

        admitted = num(row.iloc[5])
        result["spo_applications_after_9"] += num(row.iloc[3]) + num(row.iloc[4])
        result["spo_admitted_after_9"] += admitted
        result["spo_budget_after_9"] += sum(num(row.iloc[j]) for j in [9, 11, 13])
        result["spo_paid_after_9"] += num(row.iloc[14])
        result["spo_admitted_specialists_after_9"] += admitted if is_specialist else 0
        result["spo_admitted_workers_after_9"] += admitted if is_worker else 0

    return result

In [ ]:
spo_rows = []

for year_code in year_codes:
    path = spo_total_file(year_code)
    if path is None:
        continue
    spo_rows.append(
        {
            "year_code": year_code,
            "academic_year": academic_year_label(year_code),
            "year_start": academic_year_start(year_code),
            "source_file": str(path.relative_to(ROOT)),
            **read_spo_file(path),
        }
    )

spo = pd.DataFrame(spo_rows).sort_values("year_start")

school_spo = school_total.merge(
    spo, on=["year_code", "academic_year", "year_start"], how="inner"
)
school_spo["spo_admission_vs_grade9_certificates_pct"] = (
    school_spo["spo_admitted_after_9"] / school_spo["grade_9_certificates"] * 100
)
school_spo["spo_budget_share_pct"] = (
    school_spo["spo_budget_after_9"] / school_spo["spo_admitted_after_9"] * 100
)
school_spo["spo_paid_share_pct"] = (
    school_spo["spo_paid_after_9"] / school_spo["spo_admitted_after_9"] * 100
)

school_spo.tail()

,year_code,academic_year,year_start,sector,settlement,source_file_x,grade_9_students,grade_10_students,grade_11_students,grade_9_certificates,grade_9_continue_same_org,grade_11_certificates,same_school_continuation_pct,prev_grade_9_students,cohort_grade10_vs_prev_grade9_pct,source_file_y,spo_applications_after_9,spo_admitted_after_9,spo_budget_after_9,spo_paid_after_9,spo_admitted_specialists_after_9,spo_admitted_workers_after_9,spo_admission_vs_grade9_certificates_pct,spo_budget_share_pct,spo_paid_share_pct
2,21-22,2021/2022,2021,ГОУ+НОУ,город+село,21-22/ОО-1_Россия_(ГОУ+НОУ)_(город+село).xlsx,"1,464,572.00","700,761.00","643,823.00","1,393,805.00","568,449.00","675,623.00",40.78,"1,420,507.00",49.33,_extracted/21-22/Своды_СПО-1_2021/СВОД_СПО1_ВС...,"1,934,732.00","823,672.00","564,664.00","259,008.00","641,812.00","181,860.00",59.10,68.55,31.45
3,22-23,2022/2023,2022,ГОУ+НОУ,город+село,22-23/Россия_(ГОУ+НОУ)_(город+село).xlsx,"1,589,478.00","686,538.00","619,517.00","1,419,901.00","560,369.00","631,945.00",39.47,"1,464,572.00",46.88,_extracted/22-23/Своды_СПО-1_2022/СВОД_СПО1_ВС...,"2,237,358.00","895,610.00","602,823.00","292,787.00","711,679.00","183,931.00",63.08,67.31,32.69
4,23-24,2023/2024,2023,ГОУ+НОУ,город+село,23-24/Россия_(ГОУ+НОУ)_(город+село).xlsx,"1,689,558.00","719,073.00","605,111.00","1,527,664.00","583,859.00","606,147.00",38.22,"1,589,478.00",45.24,_extracted/23-24/Своды_СПО-1/СВОД_СПО1_ВСЕГО.xls,"2,542,770.00","964,947.00","626,939.00","338,008.00","778,680.00","186,267.00",63.16,64.97,35.03
5,24-25,2024/2025,2024,ГОУ+НОУ,город+село,24-25/Россия_(ГОУ+НОУ)_(город+село).xlsx,"1,741,257.00","738,721.00","632,090.00","1,622,200.00","605,428.00","593,616.00",37.32,"1,689,558.00",43.72,_extracted/24-25/Своды_СПО-1_за_2024_г/СВОД_СП...,"2,844,453.00","1,039,575.00","655,719.00","383,856.00","844,397.00","195,178.00",64.08,63.08,36.92
6,25-26,2025/2026,2025,ГОУ+НОУ,город+село,25-26/Россия_(ГОУ+НОУ)_(город+село).xlsx,"1,772,997.00","742,835.00","651,580.00","1,671,774.00","618,690.00","619,903.00",37.01,"1,741,257.00",42.66,25-26/СПО -1 на начало 2025_26 учебный год/СВО...,"3,548,599.00","1,121,001.00","721,717.00","399,284.00","910,136.00","210,865.00",67.05,64.38,35.62


### 2.3. Вузы: форма ВПО-1

In [ ]:
vpo_years = sorted(
    int(re.search(r"VPO_1_(\d{4})", p.name).group(1)) for p in VPO_ROOT.glob("VPO_1_*")
)


def vpo_file(year: int, form: str) -> Path:
    files = []
    for path in (VPO_ROOT / f"VPO_1_{year}").rglob("*.xls*"):
        name = path.name.lower()
        if not name.startswith("свод_впо1_всего"):
            continue
        if form == "total" and not any(
            word in name
            for word in [
                "очная",
                "заочная",
                "очно",
                "персонал",
                "сведения",
                "аттеста",
                "гос",
                "негос",
            ]
        ):
            files.append(path)
        if (
            form == "fulltime"
            and "очная" in name
            and "очно-заочная" not in name
            and "заочная" not in name
        ):
            files.append(path)
    return sorted(files, key=lambda p: (len(str(p)), str(p)))[0]


def vpo_sheet(path: Path, sheet_code: str) -> str:
    code = sheet_code.replace(" ", "").lower()
    for sheet in pd.ExcelFile(path).sheet_names:
        name = sheet.replace("P", "Р").replace("p", "р").replace(" ", "").lower()
        if code in name:
            return sheet
    raise ValueError(f"Лист {sheet_code} не найден в {path.name}")


def vpo_columns(df: pd.DataFrame) -> tuple[int, list[int], int, int, int, int | None]:
    # В новых ВПО-1 появились отдельные графы для квот, поэтому номера колонок сдвинулись.
    if df.shape[1] >= 19:
        return 8, [10, 12, 13], 17, 14, 15, 16
    return 7, [9, 11, 12], 15, 13, 14, None


def program_total_rows(df: pd.DataFrame) -> pd.DataFrame:
    labels = df.iloc[:, 0].fillna("").astype(str)
    mask = labels.str.contains(
        "Программы бакалавриата - всего|Программы специалитета - всего|Программы магистратуры - всего",
        regex=True,
    )
    return df[mask]


def read_vpo_admission(path: Path) -> dict[str, float]:
    df = pd.read_excel(path, sheet_name=vpo_sheet(path, "Р2_1_1"), header=None)
    admitted_col, budget_cols, paid_col, target_col, special_col, separate_col = (
        vpo_columns(df)
    )
    rows = program_total_rows(df)

    return {
        "vpo_admitted_total": rows.iloc[:, admitted_col].map(num).sum(),
        "vpo_budget_total": sum(rows.iloc[:, j].map(num).sum() for j in budget_cols),
        "vpo_paid_total": rows.iloc[:, paid_col].map(num).sum(),
        "vpo_bachelor_admitted": rows[
            rows.iloc[:, 0].astype(str).str.contains("бакалавриата")
        ]
        .iloc[:, admitted_col]
        .map(num)
        .sum(),
        "vpo_specialist_admitted": rows[
            rows.iloc[:, 0].astype(str).str.contains("специалитета")
        ]
        .iloc[:, admitted_col]
        .map(num)
        .sum(),
        "vpo_master_admitted": rows[
            rows.iloc[:, 0].astype(str).str.contains("магистратуры")
        ]
        .iloc[:, admitted_col]
        .map(num)
        .sum(),
        "vpo_target_quota_total": rows.iloc[:, target_col].map(num).sum(),
        "vpo_special_quota_total": rows.iloc[:, special_col].map(num).sum(),
        "vpo_separate_quota_total": rows.iloc[:, separate_col].map(num).sum()
        if separate_col is not None
        else 0.0,
    }

In [ ]:
vpo_rows = []
vpo_fulltime_rows = []

for year in vpo_years:
    total_path = vpo_file(year, "total")
    fulltime_path = vpo_file(year, "fulltime")
    vpo_rows.append(
        {
            "year": year,
            "source_file": str(total_path.relative_to(ROOT)),
            **read_vpo_admission(total_path),
        }
    )
    vpo_fulltime_rows.append(
        {
            "year": year,
            "source_file": str(fulltime_path.relative_to(ROOT)),
            **read_vpo_admission(fulltime_path),
        }
    )

vpo = pd.DataFrame(vpo_rows).sort_values("year")
vpo_fulltime = pd.DataFrame(vpo_fulltime_rows).sort_values("year")

for df in [vpo, vpo_fulltime]:
    df["vpo_budget_share_pct"] = df["vpo_budget_total"] / df["vpo_admitted_total"] * 100
    df["vpo_paid_share_pct"] = df["vpo_paid_total"] / df["vpo_admitted_total"] * 100
    df["vpo_bachelor_share_pct"] = (
        df["vpo_bachelor_admitted"] / df["vpo_admitted_total"] * 100
    )
    df["vpo_specialist_share_pct"] = (
        df["vpo_specialist_admitted"] / df["vpo_admitted_total"] * 100
    )
    df["vpo_master_share_pct"] = (
        df["vpo_master_admitted"] / df["vpo_admitted_total"] * 100
    )

vpo.tail()

,year,source_file,vpo_admitted_total,vpo_budget_total,vpo_paid_total,vpo_bachelor_admitted,vpo_specialist_admitted,vpo_master_admitted,vpo_target_quota_total,vpo_special_quota_total,vpo_separate_quota_total,vpo_budget_share_pct,vpo_paid_share_pct,vpo_bachelor_share_pct,vpo_specialist_share_pct,vpo_master_share_pct
4,2021,_extracted/VPO_1_2021/СВОД_ВПО1_ВСЕГО.xls,"1,129,097.00","554,151.00","574,946.00","732,139.00","176,364.00","220,594.00","45,168.00","16,981.00",0.00,49.08,50.92,64.84,15.62,19.54
5,2022,_extracted/VPO_1_2022/СВОД_ВПО1_ВСЕГО.xls,"1,201,541.00","592,081.00","609,460.00","751,832.00","187,551.00","262,158.00","45,035.00","17,152.00",0.00,49.28,50.72,62.57,15.61,21.82
6,2023,_extracted/VPO_1_2023/СВОД_ВПО1_ВСЕГО.xls,"1,287,601.00","565,203.00","722,398.00","819,626.00","202,366.00","265,609.00","47,341.00","16,904.00","8,730.00",43.90,56.10,63.66,15.72,20.63
7,2024,_extracted/VPO_1_2024/СВОД_ВПО1_ВСЕГО.xls,"1,298,067.00","566,136.00","731,931.00","829,713.00","208,337.00","260,017.00","42,807.00","16,730.00","15,259.00",43.61,56.39,63.92,16.05,20.03
8,2025,_extracted/VPO_1_2025/СВОД_ВПО1_ВСЕГО.xlsx,"1,419,459.00","567,945.00","851,514.00","905,472.00","224,459.00","289,528.00","49,327.00","17,880.00","28,004.00",40.01,59.99,63.79,15.81,20.40


In [ ]:
def read_vpo_directions(path: Path, year: int) -> list[dict]:
    df = pd.read_excel(path, sheet_name=vpo_sheet(path, "Р2_1_1"), header=None)
    admitted_col, budget_cols, paid_col, target_col, special_col, separate_col = (
        vpo_columns(df)
    )
    applications_paid_col = 7 if df.shape[1] >= 19 else 6

    rows = []
    program_level = None
    for _, row in df.iterrows():
        label = clean_text(row.iloc[0])
        label_original = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ""

        if "программы бакалавриата" in label:
            program_level = "бакалавриат"
        elif "программы специалитета" in label:
            program_level = "специалитет"
        elif "программы магистратуры" in label:
            program_level = "магистратура"

        code = str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else ""
        if "всего" in label or not re.fullmatch(r"\d{2}\.\d{2}\.\d{2}", code):
            continue

        admitted = num(row.iloc[admitted_col])
        if admitted == 0:
            continue

        budget = sum(num(row.iloc[j]) for j in budget_cols)
        paid = num(row.iloc[paid_col])
        applications = num(row.iloc[3]) + num(row.iloc[applications_paid_col])

        rows.append(
            {
                "year": year,
                "program_level": program_level,
                "direction_code": code,
                "direction_name": label_original,
                "applications_total": applications,
                "admitted_total": admitted,
                "budget_total": budget,
                "paid_total": paid,
                "target_quota_total": num(row.iloc[target_col]),
                "special_quota_total": num(row.iloc[special_col]),
                "separate_quota_total": num(row.iloc[separate_col])
                if separate_col is not None
                else 0.0,
                "applications_per_admitted": applications / admitted,
                "budget_share_pct": budget / admitted * 100,
                "paid_share_pct": paid / admitted * 100,
            }
        )
    return rows


direction_rows = []
for year in vpo_years:
    direction_rows.extend(read_vpo_directions(vpo_file(year, "fulltime"), year))

vpo_directions = pd.DataFrame(direction_rows)
vpo_directions.tail()

,year,program_level,direction_code,direction_name,applications_total,admitted_total,budget_total,paid_total,target_quota_total,special_quota_total,separate_quota_total,applications_per_admitted,budget_share_pct,paid_share_pct
5243,2025,магистратура,44.04.03,Специальное (дефектологическое) образование,33.00,9.00,9.00,0.00,1.00,0.00,0.00,3.67,100.00,0.00
5244,2025,магистратура,45.04.01,Филология,223.00,98.00,82.00,16.00,0.00,0.00,0.00,2.28,83.67,16.33
5245,2025,магистратура,45.04.02,Лингвистика,116.00,33.00,28.00,5.00,0.00,0.00,0.00,3.52,84.85,15.15
5246,2025,магистратура,52.04.03,Театральное искусство,8.00,1.00,0.00,1.00,0.00,0.00,0.00,8.00,0.00,100.00
5247,2025,магистратура,54.04.01,Дизайн,54.00,19.00,1.00,18.00,0.00,0.00,0.00,2.84,5.26,94.74


In [ ]:
def printed_columns(df: pd.DataFrame) -> dict[int, int]:
    for i in range(25):
        mapping = {}
        for j, value in enumerate(df.iloc[i]):
            text = str(value).strip()
            if re.fullmatch(r"\d+", text):
                mapping[int(text)] = j
        if 54 in mapping and 56 in mapping:
            return mapping
    raise ValueError("Не найдены печатные номера граф")


def read_vpo_students(path: Path) -> dict[str, float]:
    try:
        sheet = vpo_sheet(path, "Р2_1_2(4)")
    except ValueError:
        sheet = vpo_sheet(path, "Р2_1_2")
    df = pd.read_excel(path, sheet_name=sheet, header=None)
    cols = printed_columns(df)
    total_row = df[
        df.iloc[:, 0].fillna("").astype(str).str.startswith("Всего по программам")
    ].iloc[-1]

    return {
        "vpo_students_total": num(total_row.iloc[cols[54]]),
        "vpo_students_budget_total": sum(
            num(total_row.iloc[cols[c]]) for c in [56, 58, 59] if c in cols
        ),
        "vpo_students_paid_total": num(total_row.iloc[cols.get(62, cols.get(61))]),
    }


student_rows = []
for year in vpo_years:
    path = vpo_file(year, "total")
    student_rows.append(
        {
            "year": year,
            "source_file": str(path.relative_to(ROOT)),
            **read_vpo_students(path),
        }
    )

vpo_students = pd.DataFrame(student_rows).sort_values("year")
vpo_students["vpo_students_budget_share_pct"] = (
    vpo_students["vpo_students_budget_total"] / vpo_students["vpo_students_total"] * 100
)
vpo_students["vpo_students_paid_share_pct"] = (
    vpo_students["vpo_students_paid_total"] / vpo_students["vpo_students_total"] * 100
)
vpo_students.tail()

,year,source_file,vpo_students_total,vpo_students_budget_total,vpo_students_paid_total,vpo_students_budget_share_pct,vpo_students_paid_share_pct
4,2021,_extracted/VPO_1_2021/СВОД_ВПО1_ВСЕГО.xls,"4,044,203.00","1,934,641.00","2,118,193.00",47.84,52.38
5,2022,_extracted/VPO_1_2022/СВОД_ВПО1_ВСЕГО.xls,"4,130,018.00","2,008,296.00","2,138,242.00",48.63,51.77
6,2023,_extracted/VPO_1_2023/СВОД_ВПО1_ВСЕГО.xls,"4,325,280.00","2,051,732.00","2,273,548.00",47.44,52.56
7,2024,_extracted/VPO_1_2024/СВОД_ВПО1_ВСЕГО.xls,"4,431,687.00","2,067,167.00","2,364,520.00",46.65,53.35
8,2025,_extracted/VPO_1_2025/СВОД_ВПО1_ВСЕГО.xlsx,"4,657,959.00","2,077,163.00","2,580,796.00",44.59,55.41


In [ ]:
population_rows = []

for path in sorted(ROOT.glob("Chisl_polvozr_01-01-*_VPN-2020.xlsx")):
    year = int(re.search(r"01-01-(\d{4})", path.name).group(1))
    df = pd.read_excel(path, sheet_name="М+Ж_Г+С", header=None)

    header_row = next(
        i
        for i in range(12)
        if "17" in {str(v).strip() for v in df.iloc[i] if pd.notna(v)}
        or "17.0" in {str(v).strip() for v in df.iloc[i] if pd.notna(v)}
    )
    age_columns = []
    for j, value in enumerate(df.iloc[header_row]):
        try:
            age = int(float(str(value).replace(",", ".")))
        except ValueError:
            continue
        if 17 <= age <= 30:
            age_columns.append(j)

    rf_row = df[
        df.iloc[:, 0].fillna("").astype(str).str.strip().eq("Российская Федерация")
    ].iloc[0]
    population_rows.append(
        {
            "year": year,
            "population_17_30": sum(num(rf_row.iloc[j]) for j in age_columns),
            "source_file": path.name,
        }
    )

population = pd.DataFrame(population_rows).sort_values("year")

vpo_guarantee = vpo_students.merge(population, on="year", how="left")
vpo_guarantee["students_per_10k_population_17_30"] = (
    vpo_guarantee["vpo_students_total"] / vpo_guarantee["population_17_30"] * 10_000
)
vpo_guarantee["budget_students_per_10k_population_17_30"] = (
    vpo_guarantee["vpo_students_budget_total"]
    / vpo_guarantee["population_17_30"]
    * 10_000
)
vpo_guarantee["guarantee_800_met"] = (
    vpo_guarantee["budget_students_per_10k_population_17_30"]
    .ge(800)
    .where(
        vpo_guarantee["budget_students_per_10k_population_17_30"].notna(),
        pd.NA,
    )
)

vpo_guarantee[
    [
        "year",
        "vpo_students_budget_total",
        "population_17_30",
        "budget_students_per_10k_population_17_30",
        "guarantee_800_met",
    ]
]

,year,vpo_students_budget_total,population_17_30,budget_students_per_10k_population_17_30,guarantee_800_met
0,2017,"1,904,603.00","26,398,098.00",721.49,False
1,2018,"1,911,728.00","25,309,734.00",755.33,False
2,2019,"1,890,108.00","24,129,078.00",783.33,False
3,2020,"1,904,577.00","23,243,386.00",819.41,True
4,2021,"1,934,641.00","22,373,052.00",864.72,True
5,2022,"2,008,296.00","21,641,199.00",928.00,True
6,2023,"2,051,732.00",NaN,NaN,<NA>
7,2024,"2,067,167.00",NaN,NaN,<NA>
8,2025,"2,077,163.00",NaN,NaN,<NA>


In [12]:
organization_rows = []

for path in sorted(VPO_ROOT.glob("VPO_1_*/**/Статистика*ВПО-1*.xlsx")):
    year = int(re.search(r"VPO_1_(\d{4})", str(path)).group(1))
    df = pd.read_excel(path, sheet_name=0, header=None)
    record = {"year": year}
    for _, row in df.iterrows():
        label = clean_text(row.iloc[0])
        if label == "всего":
            record["organizations_total"] = num(row.iloc[1])
        elif label == "государственные":
            record["state_organizations"] = num(row.iloc[1])
        elif label == "негосударственные":
            record["nonstate_organizations"] = num(row.iloc[1])
    organization_rows.append(record)

vpo_organizations = pd.DataFrame(organization_rows).sort_values("year")

pd.DataFrame(
    [
        ["school", *school.shape],
        ["school_spo", *school_spo.shape],
        ["vpo", *vpo.shape],
        ["vpo_fulltime", *vpo_fulltime.shape],
        ["vpo_directions", *vpo_directions.shape],
        ["vpo_students", *vpo_students.shape],
        ["vpo_guarantee", *vpo_guarantee.shape],
    ],
    columns=["таблица", "строк", "колонок"],
)

,таблица,строк,колонок
0,school,63,15
1,school_spo,7,25
2,vpo,9,16
3,vpo_fulltime,9,16
4,vpo_directions,5248,14
5,vpo_students,9,7
6,vpo_guarantee,9,12


## 3. Функции для графиков

In [ ]:
def trend_values(
    df: pd.DataFrame, x_col: str, y_col: str, degree: int = 1
) -> pd.Series:
    data = df[[x_col, y_col]].dropna().sort_values(x_col)
    if len(data) <= degree:
        return pd.Series(index=df.index, dtype=float)
    x = data[x_col].to_numpy(dtype=float)
    y = data[y_col].to_numpy(dtype=float)
    center = x.mean()
    coeff = np.polyfit(x - center, y, degree)
    return pd.Series(
        np.polyval(coeff, df[x_col].to_numpy(dtype=float) - center), index=df.index
    )


def finish(
    fig: go.Figure, title: str, x_title: str = "", y_title: str = "", height: int = 520
) -> None:
    fig.update_layout(
        title=title,
        height=height,
        font={"family": "Arial", "size": 14, "color": "#1f2933"},
        legend={
            "orientation": "h",
            "yanchor": "bottom",
            "y": -0.25,
            "xanchor": "left",
            "x": 0,
        },
        margin={"l": 60, "r": 30, "t": 80, "b": 90},
        xaxis_title=x_title,
        yaxis_title=y_title,
    )
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(gridcolor="#ddd6cc")
    fig.show(config={"displayModeBar": False})


def fact_line(
    fig: go.Figure,
    df: pd.DataFrame,
    x: str,
    y: str,
    name: str,
    color: str,
    suffix: str = "",
) -> None:
    fig.add_trace(
        go.Scatter(
            x=df[x],
            y=df[y],
            mode="lines+markers+text",
            name=name,
            line={"color": color, "width": 2.2, "dash": "dash"},
            marker={"size": 8},
            text=[f"{v:.1f}{suffix}" if pd.notna(v) else "" for v in df[y]],
            textposition="top center",
            hovertemplate="%{x}<br>%{y:.1f}" + suffix + f"<extra>{name}</extra>",
        )
    )


def trend_line(
    fig: go.Figure,
    df: pd.DataFrame,
    x_num: str,
    y: str,
    x_show: str,
    name: str,
    color: str,
    degree: int = 1,
) -> None:
    work = df.copy()
    work["_trend"] = trend_values(work, x_num, y, degree=degree)
    fig.add_trace(
        go.Scatter(
            x=work[x_show],
            y=work["_trend"],
            mode="lines",
            name=name,
            line={"color": color, "width": 3.4},
            hovertemplate="%{x}<br>%{y:.1f}<extra>" + name + "</extra>",
        )
    )


def pct(value: float) -> str:
    return f"{value:.1f}%".replace(".", ",")


def whole(value: float) -> str:
    return f"{value:,.0f}".replace(",", " ")

## 4. Задача 1. Траектории после 9 класса

In [ ]:
task1 = school_spo.sort_values("year_start").copy()

task1[
    [
        "academic_year",
        "grade_9_certificates",
        "grade_9_continue_same_org",
        "spo_admitted_after_9",
        "same_school_continuation_pct",
        "spo_admission_vs_grade9_certificates_pct",
        "spo_budget_share_pct",
        "spo_paid_share_pct",
    ]
].round(1)

,academic_year,grade_9_certificates,grade_9_continue_same_org,spo_admitted_after_9,same_school_continuation_pct,spo_admission_vs_grade9_certificates_pct,spo_budget_share_pct,spo_paid_share_pct
0,2019/2020,"1,406,033.00","611,556.00","782,476.00",43.50,55.70,70.30,29.70
1,2020/2021,"1,463,542.00","590,394.00","842,584.00",40.30,57.60,67.40,32.60
2,2021/2022,"1,393,805.00","568,449.00","823,672.00",40.80,59.10,68.60,31.40
3,2022/2023,"1,419,901.00","560,369.00","895,610.00",39.50,63.10,67.30,32.70
4,2023/2024,"1,527,664.00","583,859.00","964,947.00",38.20,63.20,65.00,35.00
5,2024/2025,"1,622,200.00","605,428.00","1,039,575.00",37.30,64.10,63.10,36.90
6,2025/2026,"1,671,774.00","618,690.00","1,121,001.00",37.00,67.10,64.40,35.60


In [ ]:
fig = go.Figure()
fact_line(
    fig,
    task1,
    "academic_year",
    "same_school_continuation_pct",
    "10 класс той же школы, факт",
    COLORS["red"],
    "%",
)
fact_line(
    fig,
    task1,
    "academic_year",
    "spo_admission_vs_grade9_certificates_pct",
    "СПО после 9 класса, факт",
    COLORS["teal"],
    "%",
)
trend_line(
    fig,
    task1,
    "year_start",
    "same_school_continuation_pct",
    "academic_year",
    "тренд 10 класса",
    COLORS["red"],
)
trend_line(
    fig,
    task1,
    "year_start",
    "spo_admission_vs_grade9_certificates_pct",
    "academic_year",
    "тренд СПО",
    COLORS["teal"],
)
fig.update_yaxes(ticksuffix="%", range=[30, 75])
finish(
    fig,
    "После 9 класса: 10 класс той же школы и поступление в СПО",
    "Учебный год",
    "% от получивших аттестат за 9 класс",
)

In [ ]:
abs_task1 = task1.assign(
    grade_9_mln=task1["grade_9_certificates"] / 1_000_000,
    grade10_mln=task1["grade_9_continue_same_org"] / 1_000_000,
    spo_mln=task1["spo_admitted_after_9"] / 1_000_000,
)

fig = go.Figure()
fact_line(
    fig,
    abs_task1,
    "academic_year",
    "grade_9_mln",
    "аттестаты за 9 класс",
    COLORS["blue"],
)
fact_line(
    fig,
    abs_task1,
    "academic_year",
    "grade10_mln",
    "10 класс той же школы",
    COLORS["red"],
)
fact_line(
    fig, abs_task1, "academic_year", "spo_mln", "приём в СПО после 9", COLORS["teal"]
)
finish(
    fig, "Абсолютные масштабы траекторий после 9 класса", "Учебный год", "млн человек"
)

In [ ]:
fig = go.Figure()
fig.add_bar(
    x=task1["academic_year"],
    y=task1["spo_budget_share_pct"],
    name="бюджет",
    marker_color=COLORS["green"],
    text=[pct(v) for v in task1["spo_budget_share_pct"]],
    textposition="inside",
)
fig.add_bar(
    x=task1["academic_year"],
    y=task1["spo_paid_share_pct"],
    name="платное",
    marker_color=COLORS["gold"],
    text=[pct(v) for v in task1["spo_paid_share_pct"]],
    textposition="inside",
)
fig.update_layout(barmode="stack")
fig.update_yaxes(ticksuffix="%", range=[0, 100])
finish(
    fig, "СПО после 9 класса: бюджетный и платный приём", "Учебный год", "% от приёма"
)

In [ ]:
spo_programs = task1.assign(
    specialist_share=task1["spo_admitted_specialists_after_9"]
    / task1["spo_admitted_after_9"]
    * 100,
    worker_share=task1["spo_admitted_workers_after_9"]
    / task1["spo_admitted_after_9"]
    * 100,
)

fig = go.Figure()
fig.add_bar(
    x=spo_programs["academic_year"],
    y=spo_programs["specialist_share"],
    name="специалисты среднего звена",
    marker_color=COLORS["purple"],
    text=[pct(v) for v in spo_programs["specialist_share"]],
    textposition="inside",
)
fig.add_bar(
    x=spo_programs["academic_year"],
    y=spo_programs["worker_share"],
    name="рабочие и служащие",
    marker_color=COLORS["blue"],
    text=[pct(v) for v in spo_programs["worker_share"]],
    textposition="inside",
)
fig.update_layout(barmode="stack")
fig.update_yaxes(ticksuffix="%", range=[0, 100])
finish(fig, "Структура приёма в СПО после 9 класса", "Учебный год", "% от приёма")

In [ ]:
city_rural = school.query(
    "sector == 'ГОУ+НОУ' and settlement in ['город', 'село']"
).sort_values("year_start")

fig = px.line(
    city_rural,
    x="academic_year",
    y="same_school_continuation_pct",
    color="settlement",
    markers=True,
    labels={
        "academic_year": "Учебный год",
        "same_school_continuation_pct": "%",
        "settlement": "Тип территории",
    },
    color_discrete_map={"город": COLORS["blue"], "село": COLORS["green"]},
)
fig.update_traces(line_dash="dash", line_width=2.2, marker_size=8)
fig.update_yaxes(ticksuffix="%", range=[25, 50])
finish(fig, "Переход в 10 класс той же школы: город и село", "Учебный год", "%")

**Вывод по задаче 1.** После 9 класса усиливается траектория среднего профессионального образования. Доля поступивших в СПО растёт, а доля тех, кто продолжает обучение в 10 классе той же школы, снижается. В 2025/2026 учебном году в СПО после 9 класса принято примерно `1,12 млн` человек, или `67,1%` от числа получивших аттестат за 9 класс. При этом колледжи остаются преимущественно бюджетной траекторией: бюджетная доля приёма равна `64,4%`.

## 5. Задача 2. Доступность высшего образования

In [ ]:
primary = (
    vpo_directions[vpo_directions["program_level"].isin(["бакалавриат", "специалитет"])]
    .groupby("year", as_index=False)[
        [
            "admitted_total",
            "budget_total",
            "paid_total",
            "target_quota_total",
            "special_quota_total",
            "separate_quota_total",
        ]
    ]
    .sum()
)
primary["budget_share_pct"] = primary["budget_total"] / primary["admitted_total"] * 100
primary["paid_share_pct"] = primary["paid_total"] / primary["admitted_total"] * 100
primary["year_label"] = primary["year"].astype(str)

master = (
    vpo_directions[vpo_directions["program_level"].eq("магистратура")]
    .groupby("year", as_index=False)[["admitted_total", "budget_total", "paid_total"]]
    .sum()
)
master["budget_share_pct"] = master["budget_total"] / master["admitted_total"] * 100
master["paid_share_pct"] = master["paid_total"] / master["admitted_total"] * 100
master["year_label"] = master["year"].astype(str)

primary.tail()

,year,admitted_total,budget_total,paid_total,target_quota_total,special_quota_total,separate_quota_total,budget_share_pct,paid_share_pct,year_label
3,2021,"589,513.00","364,953.00","224,560.00","38,497.00","15,112.00",0.00,61.91,38.09,2021
4,2022,"607,413.00","369,276.00","238,137.00","38,377.00","15,074.00",0.00,60.79,39.21,2022
5,2023,"621,220.00","372,219.00","249,001.00","40,199.00","14,756.00","7,464.00",59.92,40.08,2023
6,2024,"647,063.00","374,785.00","272,278.00","37,566.00","14,292.00","12,513.00",57.92,42.08,2024
7,2025,"672,102.00","375,351.00","296,751.00","42,775.00","15,099.00","23,620.00",55.85,44.15,2025


In [ ]:
fig = go.Figure()
fact_line(
    fig, primary, "year_label", "budget_share_pct", "бюджет, факт", COLORS["green"], "%"
)
fact_line(
    fig, primary, "year_label", "paid_share_pct", "платное, факт", COLORS["gold"], "%"
)
trend_line(
    fig,
    primary,
    "year",
    "budget_share_pct",
    "year_label",
    "тренд бюджета",
    COLORS["green"],
)
fig.update_yaxes(ticksuffix="%", range=[35, 65])
finish(
    fig,
    "Очный бакалавриат + специалитет: бюджет и платное обучение",
    "Год ВПО-1",
    "% от приёма",
)

In [ ]:
budget_abs = (
    vpo[["year", "vpo_budget_total"]]
    .merge(
        vpo_fulltime[["year", "vpo_budget_total"]].rename(
            columns={"vpo_budget_total": "fulltime_budget_total"}
        ),
        on="year",
    )
    .merge(
        primary[["year", "budget_total"]].rename(
            columns={"budget_total": "primary_fulltime_budget_total"}
        ),
        on="year",
    )
)

budget_abs["all_vpo_budget_thousand"] = budget_abs["vpo_budget_total"] / 1000
budget_abs["fulltime_budget_thousand"] = budget_abs["fulltime_budget_total"] / 1000
budget_abs["primary_budget_thousand"] = (
    budget_abs["primary_fulltime_budget_total"] / 1000
)

fig = go.Figure()
fact_line(
    fig, budget_abs, "year", "all_vpo_budget_thousand", "весь ВПО-1", COLORS["blue"]
)
fact_line(
    fig, budget_abs, "year", "fulltime_budget_thousand", "очная форма", COLORS["teal"]
)
fact_line(
    fig,
    budget_abs,
    "year",
    "primary_budget_thousand",
    "очный бакалавриат+специалитет",
    COLORS["green"],
)
fig.update_yaxes(range=[250, 650])
finish(fig, "Бюджетный приём в вузах: абсолютные значения", "Год ВПО-1", "тыс. человек")

In [ ]:
forms = (
    vpo[["year", "vpo_admitted_total"]]
    .rename(columns={"vpo_admitted_total": "total"})
    .merge(
        vpo_fulltime[["year", "vpo_admitted_total"]].rename(
            columns={"vpo_admitted_total": "fulltime"}
        ),
        on="year",
    )
)
forms = forms[forms["fulltime"] > 0].copy()
forms["non_fulltime"] = forms["total"] - forms["fulltime"]

fig = go.Figure()
fig.add_bar(
    x=forms["year"],
    y=forms["fulltime"] / 1000,
    name="очная форма",
    marker_color=COLORS["blue"],
    text=(forms["fulltime"] / 1000).round(1),
    textposition="inside",
)
fig.add_bar(
    x=forms["year"],
    y=forms["non_fulltime"] / 1000,
    name="очно-заочная и заочная",
    marker_color=COLORS["gold"],
    text=(forms["non_fulltime"] / 1000).round(1),
    textposition="inside",
)
fig.update_layout(barmode="stack")
finish(fig, "Весь приём ВПО-1 по формам обучения", "Год ВПО-1", "тыс. человек")

In [ ]:
fig = go.Figure()
fact_line(
    fig,
    master,
    "year_label",
    "budget_share_pct",
    "бюджет магистратуры, факт",
    COLORS["purple"],
    "%",
)
fact_line(
    fig, master, "year_label", "paid_share_pct", "платное, факт", COLORS["gold"], "%"
)
trend_line(
    fig,
    master,
    "year",
    "budget_share_pct",
    "year_label",
    "тренд бюджета магистратуры",
    COLORS["purple"],
)
fig.update_yaxes(ticksuffix="%", range=[0, 90])
finish(
    fig,
    "Очная магистратура отдельно от школьной траектории",
    "Год ВПО-1",
    "% от приёма",
)

In [ ]:
fig = go.Figure()
fig.add_bar(
    x=primary["year"],
    y=primary["target_quota_total"] / 1000,
    name="целевая квота",
    marker_color=COLORS["teal"],
    text=(primary["target_quota_total"] / 1000).round(1),
    textposition="inside",
)
fig.add_bar(
    x=primary["year"],
    y=primary["special_quota_total"] / 1000,
    name="особая/установленная квота",
    marker_color=COLORS["purple"],
    text=(primary["special_quota_total"] / 1000).round(1),
    textposition="inside",
)
fig.add_bar(
    x=primary["year"],
    y=primary["separate_quota_total"] / 1000,
    name="отдельная квота",
    marker_color=COLORS["red"],
    text=(primary["separate_quota_total"] / 1000).round(1),
    textposition="inside",
)
fig.update_layout(barmode="stack")
finish(
    fig,
    "Квоты в очном первичном приёме: бакалавриат + специалитет",
    "Год ВПО-1",
    "тыс. человек",
)

In [ ]:
school11 = school_total[["academic_year", "year_start", "grade_11_certificates"]].copy()
school_vpo = school11.merge(
    primary, left_on="year_start", right_on="year", how="inner"
).sort_values("year_start")
school_vpo["primary_admission_vs_grade11_pct"] = (
    school_vpo["admitted_total"] / school_vpo["grade_11_certificates"] * 100
)
school_vpo["primary_budget_vs_grade11_pct"] = (
    school_vpo["budget_total"] / school_vpo["grade_11_certificates"] * 100
)

school_vpo[
    [
        "academic_year",
        "grade_11_certificates",
        "admitted_total",
        "budget_total",
        "primary_admission_vs_grade11_pct",
        "primary_budget_vs_grade11_pct",
    ]
].round(1)

,academic_year,grade_11_certificates,admitted_total,budget_total,primary_admission_vs_grade11_pct,primary_budget_vs_grade11_pct
0,2019/2020,"654,710.00","549,927.00","313,075.00",84.00,47.80
1,2020/2021,"680,849.00","557,308.00","334,043.00",81.90,49.10
2,2021/2022,"675,623.00","589,513.00","364,953.00",87.30,54.00
3,2022/2023,"631,945.00","607,413.00","369,276.00",96.10,58.40
4,2023/2024,"606,147.00","621,220.00","372,219.00",102.50,61.40
5,2024/2025,"593,616.00","647,063.00","374,785.00",109.00,63.10
6,2025/2026,"619,903.00","672,102.00","375,351.00",108.40,60.50


In [ ]:
fig = go.Figure()
fact_line(
    fig,
    school_vpo,
    "academic_year",
    "primary_admission_vs_grade11_pct",
    "первичный очный приём, факт",
    COLORS["teal"],
    "%",
)
fact_line(
    fig,
    school_vpo,
    "academic_year",
    "primary_budget_vs_grade11_pct",
    "бюджетный первичный очный приём, факт",
    COLORS["green"],
    "%",
)
trend_line(
    fig,
    school_vpo,
    "year_start",
    "primary_budget_vs_grade11_pct",
    "academic_year",
    "тренд бюджетного приёма",
    COLORS["green"],
)
fig.update_yaxes(ticksuffix="%", range=[45, 190])
finish(
    fig,
    "Очный первичный приём в вузы относительно выпуска 11 класса",
    "Учебный год",
    "% от числа аттестатов 11 класса",
)

In [ ]:
latest = school_vpo.iloc[-1]
all_vpo_latest = vpo[vpo["year"].eq(int(latest["year"]))].iloc[0]

pd.DataFrame(
    {
        "показатель": [
            "аттестаты за 11 класс",
            "очный бюджет бакалавриата+специалитета",
            "очный бюджет / выпуск 11 класса",
            "весь бюджетный приём ВПО-1",
            "весь бюджетный приём ВПО-1 / выпуск 11 класса",
        ],
        "значение": [
            whole(latest["grade_11_certificates"]),
            whole(latest["budget_total"]),
            pct(latest["primary_budget_vs_grade11_pct"]),
            whole(all_vpo_latest["vpo_budget_total"]),
            pct(
                all_vpo_latest["vpo_budget_total"]
                / latest["grade_11_certificates"]
                * 100
            ),
        ],
    }
)

,показатель,значение
0,аттестаты за 11 класс,619 903
1,очный бюджет бакалавриата+специалитета,375 351
2,очный бюджет / выпуск 11 класса,"60,5%"
3,весь бюджетный приём ВПО-1,567 945
4,весь бюджетный приём ВПО-1 / выпуск 11 класса,"91,6%"


**Ответ на задачу 2.** Для сравнения с выпускниками школ корректно брать не весь ВПО-1, а очный приём в бакалавриат и специалитет. Магистратура, заочная и очно-заочная формы относятся к другим траекториям и завышают показатель доступности для выпускников 11 класса. В 2025/2026 учебном году очный бюджетный приём в бакалавриат и специалитет составляет `375 351` человек против `619 903` аттестатов за 11 класс, то есть `60,5%`. Если взять весь бюджетный приём ВПО-1, получится `91,6%`, но это уже не чистая школьная траектория.

## 6. Проверка государственной гарантии 800 бюджетных студентов

In [ ]:
guarantee_known = (
    vpo_guarantee.dropna(subset=["budget_students_per_10k_population_17_30"])
    .sort_values("year")
    .copy()
)

guarantee_known[
    [
        "year",
        "vpo_students_budget_total",
        "population_17_30",
        "budget_students_per_10k_population_17_30",
        "guarantee_800_met",
    ]
].round(1)

,year,vpo_students_budget_total,population_17_30,budget_students_per_10k_population_17_30,guarantee_800_met
0,2017,"1,904,603.00","26,398,098.00",721.50,False
1,2018,"1,911,728.00","25,309,734.00",755.30,False
2,2019,"1,890,108.00","24,129,078.00",783.30,False
3,2020,"1,904,577.00","23,243,386.00",819.40,True
4,2021,"1,934,641.00","22,373,052.00",864.70,True
5,2022,"2,008,296.00","21,641,199.00",928.00,True


In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=guarantee_known["year"],
        y=guarantee_known["budget_students_per_10k_population_17_30"],
        mode="lines+markers+text",
        name="факт",
        line={"color": COLORS["teal"], "dash": "dash", "width": 2.4},
        marker={"size": 8},
        text=[
            f"{v:.0f}"
            for v in guarantee_known["budget_students_per_10k_population_17_30"]
        ],
        textposition="top center",
    )
)
fig.add_trace(
    go.Scatter(
        x=guarantee_known["year"],
        y=[800] * len(guarantee_known),
        mode="lines",
        name="порог 800",
        line={"color": COLORS["red"], "width": 3},
    )
)
trend_line(
    fig,
    guarantee_known,
    "year",
    "budget_students_per_10k_population_17_30",
    "year",
    "линейный тренд",
    COLORS["teal"],
)
fig.update_yaxes(range=[650, 980])
finish(
    fig,
    "Бюджетные студенты на 10 тыс. человек 17-30 лет",
    "Год",
    "студентов на бюджете на 10 тыс.",
)

**Вывод по гарантии.** По условию нужно считать студентов на бюджете на 10 тыс. человек возраста 17-30 лет. В 2017-2019 годах показатель ниже 800, а с 2020 по 2022 год выше 800. В 2022 году он равен примерно `928` бюджетных студентов на 10 тыс. человек 17-30 лет. Для 2023-2025 годов возрастная структура населения в добавленных файлах отсутствует, поэтому эти годы не используются в проверке гарантии.

## 7. Дополнительная статистика

In [ ]:
latest_year = int(vpo_directions["year"].max())
latest_primary = vpo_directions[
    (vpo_directions["year"].eq(latest_year))
    & (vpo_directions["program_level"].isin(["бакалавриат", "специалитет"]))
].copy()

top_admitted = latest_primary.nlargest(15, "admitted_total")[
    [
        "direction_code",
        "direction_name",
        "admitted_total",
        "budget_total",
        "paid_total",
        "applications_per_admitted",
        "budget_share_pct",
    ]
]
top_admitted

,direction_code,direction_name,admitted_total,budget_total,paid_total,applications_per_admitted,budget_share_pct
4820,31.05.01,Лечебное дело,"49,880.00","19,279.00","30,601.00",7.31,38.65
4709,44.03.05,Педагогическое образование (с двумя профилями ...,"33,178.00","26,544.00","6,634.00",10.39,80.00
4690,40.03.01,Юриспруденция,"32,224.00","5,744.00","26,480.00",11.28,17.83
4679,38.03.01,Экономика,"27,995.00","5,384.00","22,611.00",19.03,19.23
4680,38.03.02,Менеджмент,"22,038.00","3,165.00","18,873.00",19.48,14.36
4822,31.05.03,Стоматология,"19,630.00","1,814.00","17,816.00",9.87,9.24
4591,08.03.01,Строительство,"16,992.00","13,191.00","3,801.00",10.27,77.63
4592,09.03.01,Информатика и вычислительная техника,"13,971.00","10,199.00","3,772.00",23.84,73.00
4593,09.03.02,Информационные системы и технологии,"13,908.00","8,857.00","5,051.00",22.81,63.68
4594,09.03.03,Прикладная информатика,"12,009.00","8,066.00","3,943.00",23.45,67.17


In [ ]:
plot_data = top_admitted.sort_values("admitted_total")

fig = px.bar(
    plot_data,
    x="admitted_total",
    y="direction_name",
    orientation="h",
    text="admitted_total",
    labels={"admitted_total": "зачислено, человек", "direction_name": "направление"},
)
fig.update_traces(
    marker_color=COLORS["teal"], texttemplate="%{text:,.0f}", textposition="outside"
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
finish(
    fig,
    f"Топ-15 направлений очного бакалавриата и специалитета по приёму, {latest_year}",
    "",
    "направление",
    height=650,
)

In [ ]:
competition = latest_primary[latest_primary["admitted_total"] >= 2000].nlargest(
    15, "applications_per_admitted"
)[
    [
        "direction_code",
        "direction_name",
        "applications_total",
        "admitted_total",
        "applications_per_admitted",
    ]
]
competition

,direction_code,direction_name,applications_total,admitted_total,applications_per_admitted
4687,39.03.01,Социология,"110,591.00","3,513.00",31.48
4596,10.03.01,Информационная безопасность,"156,381.00","6,197.00",25.23
4572,02.03.02,Фундаментальная информатика и информационные т...,"63,721.00","2,567.00",24.82
4592,09.03.01,Информатика и вычислительная техника,"333,009.00","13,971.00",23.84
4594,09.03.03,Прикладная информатика,"281,634.00","12,009.00",23.45
4682,38.03.04,Государственное и муниципальное управление,"202,901.00","8,889.00",22.83
4593,09.03.02,Информационные системы и технологии,"317,307.00","13,908.00",22.81
4567,01.03.02,Прикладная математика и информатика,"173,991.00","7,836.00",22.20
4595,09.03.04,Программная инженерия,"207,335.00","9,360.00",22.15
4654,27.03.05,Инноватика,"46,385.00","2,221.00",20.88


In [ ]:
plot_data = competition.sort_values("applications_per_admitted")

fig = px.bar(
    plot_data,
    x="applications_per_admitted",
    y="direction_name",
    orientation="h",
    text="applications_per_admitted",
    labels={
        "applications_per_admitted": "заявлений на одно зачисленное место",
        "direction_name": "направление",
    },
)
fig.update_traces(
    marker_color=COLORS["blue"], texttemplate="%{text:.1f}", textposition="outside"
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
finish(
    fig,
    f"Направления с самым высоким конкурсом, {latest_year}",
    "",
    "направление",
    height=650,
)

In [ ]:
fig = go.Figure()
fact_line(
    fig,
    vpo_organizations,
    "year",
    "organizations_total",
    "всего организаций",
    COLORS["blue"],
)
fact_line(
    fig,
    vpo_organizations,
    "year",
    "state_organizations",
    "государственные",
    COLORS["green"],
)
finish(
    fig, "Организации, реализующие программы высшего образования", "Год", "организаций"
)

In [ ]:
pop = population.sort_values("year").copy()
pop["population_17_30_mln"] = pop["population_17_30"] / 1_000_000

fig = go.Figure()
fact_line(
    fig, pop, "year", "population_17_30_mln", "население 17-30 лет", COLORS["blue"]
)
trend_line(
    fig, pop, "year", "population_17_30_mln", "year", "линейный тренд", COLORS["blue"]
)
fig.update_yaxes(range=[20, 28])
finish(fig, "Демографическая база высшего образования", "Год", "млн человек")

## 8. Короткая контрольная проверка ВПО-1

In [ ]:
# Проверяем ключевые суммы за последний год прямо по исходной таблице ВПО-1.
path_2025 = vpo_file(2025, "fulltime")
df_2025 = pd.read_excel(
    path_2025, sheet_name=vpo_sheet(path_2025, "Р2_1_1"), header=None
)
rows_2025 = program_total_rows(df_2025)
primary_rows_2025 = rows_2025[
    rows_2025.iloc[:, 0]
    .astype(str)
    .str.contains("бакалавриата|специалитета", regex=True)
]

admitted_col, budget_cols, paid_col, *_ = vpo_columns(df_2025)
control = pd.DataFrame(
    {
        "показатель": [
            "очный бакалавриат+специалитет: принято",
            "очный бакалавриат+специалитет: бюджет",
            "очный бакалавриат+специалитет: платное",
        ],
        "из исходного Excel": [
            primary_rows_2025.iloc[:, admitted_col].map(num).sum(),
            sum(primary_rows_2025.iloc[:, j].map(num).sum() for j in budget_cols),
            primary_rows_2025.iloc[:, paid_col].map(num).sum(),
        ],
        "в расчётной таблице": [
            primary.loc[primary["year"].eq(2025), "admitted_total"].iloc[0],
            primary.loc[primary["year"].eq(2025), "budget_total"].iloc[0],
            primary.loc[primary["year"].eq(2025), "paid_total"].iloc[0],
        ],
    }
)
control["разница"] = control["из исходного Excel"] - control["в расчётной таблице"]
control

,показатель,из исходного Excel,в расчётной таблице,разница
0,очный бакалавриат+специалитет: принято,"672,102.00","672,102.00",0.00
1,очный бакалавриат+специалитет: бюджет,"375,351.00","375,351.00",0.00
2,очный бакалавриат+специалитет: платное,"296,751.00","296,751.00",0.00


## 9. Итоговые выводы

1. После 9 класса всё больше школьников выбирает колледжи: приём в СПО растёт быстрее, чем продолжение обучения в 10 классе той же школы.
2. СПО остаётся в основном бюджетной траекторией, но платный приём тоже заметен: в 2025/2026 учебном году он составляет около `35,6%`.
3. Для анализа выпускников 11 класса нельзя смешивать бакалавриат, специалитет, магистратуру и заочные формы. Основной показатель для школьной траектории - очный бюджетный приём в бакалавриат и специалитет.
4. В 2025/2026 учебном году очный бюджетный приём в бакалавриат и специалитет равен `60,5%` от числа получивших аттестат за 11 класс.
5. Абсолютное количество фактически принятых на бюджет в вузах по последним годам почти не меняется, поэтому изменение долей в основном связано с демографией и структурой приёма.
6. Государственная гарантия `800 бюджетных студентов на 10 тыс. человек 17-30 лет` выполняется с 2020 по 2022 год; в 2017-2019 годах показатель был ниже порога.